# High-resolution evapotranspiration dataset for the Aconcagua River Basin

**Author:** Lucas Vituri Santarosa  
**Affiliation:** Water Resources and Geotechnology Laboratory | Pontificia Universidad Católica de Valparaíso (PUCV)  
**Date:** February 2026

---

### Description
This script generates a temporal series of **Actual Evapotranspiration (ETa)** using the **SAFER algorithm** with fixed coefficients and a Gap-Filling strategy.

### Usage
1. **Visualize:** Run the script to generate the layers on the interactive map.
2. **Export:** Uncomment the `task.start()` lines in the *Export* section to save data to Google Drive.

---

In [ ]:
# ==============================================================================
# 1. Start
# ==============================================================================
import ee
import geemap
import math

# Initialize Earth Engine
try:
    ee.Initialize(project="ee-lucasviturisantarosa")
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

print("Google Earth Engine initialized!")

In [ ]:
# ==============================================================================
# 2. PARAMETERS
# ==============================================================================

# Spatial Region (Aconcagua Basin)
geometry = ee.FeatureCollection('projects/ee-lucasviturisantarosa/assets/ACG').geometry()

# Time Range (Target Year 2025)
YEAR = 2025
START_DATE = f'{YEAR}-01-01'
END_DATE   = f'{YEAR}-12-31'

# SAFER Fixed Coefficients (Stable Calibration)
A_FIXED = 0.3684
B_FIXED = -0.0012

# Export Configuration
EXPORT_FOLDER_MAIN = f'SAFER_RESULTS_{YEAR}'
EXPORT_FOLDER_DAILY = f'SAFER_RESULTS_{YEAR}_DAILY_STACKS'
SCALE = 30

print(f"Processing Year: {YEAR} (Optimized Mode)")

In [ ]:
# ==============================================================================
# 3. CORE FUNCTIONS (PRE-PROCESSING)
# ==============================================================================

# --- Ancillary Data ---
era5_proj = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR').first().projection()
elevation = ee.Image('USGS/SRTMGL1_003').select('elevation').reproject(crs=era5_proj)

# --- Landsat Pre-processing (ETr Calculation) ---
def process_landsat(image):
    qa = image.select('QA_PIXEL')
    mask = qa.bitwiseAnd(1<<1).eq(0).And(qa.bitwiseAnd(1<<3).eq(0)) \
             .And(qa.bitwiseAnd(1<<4).eq(0))

    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermal = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    img_sc = image.addBands(optical, overwrite=True) \
                  .addBands(thermal, overwrite=True) \
                  .updateMask(mask)

    # Indices
    albedo = img_sc.expression(
        '(0.254*B2) + (0.149*B3) + (0.147*B4) + (0.311*B5) + (0.103*B6) + (0.036*B7)',
        {
            'B2': img_sc.select('SR_B2'), 'B3': img_sc.select('SR_B3'),
            'B4': img_sc.select('SR_B4'), 'B5': img_sc.select('SR_B5'),
            'B6': img_sc.select('SR_B6'), 'B7': img_sc.select('SR_B7')
        }
    ).rename('Albedo')

    ndvi = img_sc.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    lst_c = img_sc.select('ST_B10').subtract(273.15).rename('LST_C')

    # SAFER ETr (Evaporative Fraction)
    denom = albedo.multiply(ndvi).max(1e-6)
    X = lst_c.divide(denom)
    ETr = X.multiply(B_FIXED).add(A_FIXED).exp().rename('ETr')
    ETr = ETr.clamp(0.0, 1.2)

    return image.addBands(ETr).copyProperties(image, ["system:time_start"])

# --- ET0 Penman-Monteith (ERA5-Land) ---
def calculate_et0_pm(image):
    t_mean_c = image.select('temperature_2m').subtract(273.15)

    P = ee.Image(101.3).multiply(ee.Image(293.0).subtract(elevation.multiply(0.0065)).divide(293.0).pow(5.26))
    gamma = P.multiply(0.000665)

    es = t_mean_c.expression('0.6108 * exp((17.27 * T) / (T + 237.3))', {'T': t_mean_c})
    dew_c = image.select('dewpoint_temperature_2m').subtract(273.15)
    ea = dew_c.expression('0.6108 * exp((17.27 * T) / (T + 237.3))', {'T': dew_c})
    delta = t_mean_c.expression('4098 * (0.6108 * exp((17.27 * T) / (T + 237.3))) / ((T + 237.3)**2)', {'T': t_mean_c})

    ws_10m = image.select('u_component_of_wind_10m').hypot(image.select('v_component_of_wind_10m'))
    ws_2m = ws_10m.multiply(4.87).divide(math.log(67.8 * 10 - 5.42))

    rs = image.select('surface_solar_radiation_downwards_sum').divide(1e6)
    rn = rs.multiply(0.6)

    num = delta.multiply(rn.multiply(0.408)).add(gamma.multiply(900).divide(t_mean_c.add(273)).multiply(ws_2m).multiply(es.subtract(ea)))
    den = delta.add(gamma.multiply(ws_2m.multiply(0.34).add(1)))

    return image.addBands(num.divide(den).rename('ET0_pm'))

In [ ]:
# ==============================================================================
# 4. PREPARING COLLECTIONS
# ==============================================================================
print("1. Loading Collections...")

# Load Landsat (ETr)
ls_col = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2') \
    .merge(ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')) \
    .filterDate(f'{YEAR}-01-01', f'{YEAR}-12-31') \
    .filterBounds(geometry) \
    .map(process_landsat) \
    .select(['ETr'])

# Load ERA5 (ET0)
era5_col = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR') \
    .filterDate(START_DATE, END_DATE) \
    .map(calculate_et0_pm) \
    .select(['ET0_pm'])

In [ ]:
# ==============================================================================
# 5. PROCESSING (MONTHLY BATCHING)
# ==============================================================================
print("2. Computing Daily ETa...")

daily_collections_list = []

for m in range(1, 13):
    # A. Get Representative ETr for this Month (Median)
    etr_month_img = ls_col.filter(ee.Filter.calendarRange(m, m, 'month')).median().rename('ETr')

    # Fallback: If month is empty (clouds), use Annual Median
    fallback_annual = ls_col.median().rename('ETr')
    etr_final = etr_month_img.unmask(fallback_annual).unmask(0.0)

    # B. Filter ERA5 for this Month
    era5_month_col = era5_col.filter(ee.Filter.calendarRange(m, m, 'month'))

    # C. Map over the days of THIS month only (Client-side loop ensures manageable chunks)
    def apply_safer_daily(img_era5):
        # ETa = ET0_daily * ETr_monthly_representative
        et0 = img_era5.select('ET0_pm')
        eta = et0.multiply(etr_final).rename('ETa_mm')

        date = img_era5.date()
        date_str = date.format('YYYYMMdd')

        return ee.Image(
            eta.copyProperties(img_era5, ['system:time_start']) \
               .set('date', date.format('YYYY-MM-dd')) \
               .set('month', m) \
               .set('system:index', date_str) # Helps with naming later
        ).clip(geometry).toFloat()

    # Process the monthly chunk
    daily_col_m = era5_month_col.map(apply_safer_daily)
    daily_collections_list.append(daily_col_m)

# D. Merge monthly chunks into a single Annual Collection
daily_eta_full = ee.ImageCollection(daily_collections_list[0])
for i in range(1, 12):
    daily_eta_full = daily_eta_full.merge(daily_collections_list[i])

print("   -> Daily Series Calculated Successfully.")

In [ ]:
# ==============================================================================
# 6. AGGREGATION & EXPORT
# ==============================================================================
print("3. Aggregating and Preparing Exports...")

# --- A. Annual Sum ---
annual_eta = daily_eta_full.sum().rename('ETa_Annual_mm').clip(geometry).toFloat()

# --- B. Monthly Aggregates ---
def sum_monthly_final(m):
    return daily_eta_full.filter(ee.Filter.eq('month', m)).sum() \
                        .rename('ETa_Monthly_mm') \
                        .set('month', m) \
                        .clip(geometry).toFloat()

months = ee.List.sequence(1, 12)
monthly_eta_col = ee.ImageCollection.fromImages(months.map(sum_monthly_final))

# --- EXPORT TASKS (Fixed: Removed tileScale) ---
print("4. Defining Export Tasks...")

# 1. Daily Stacks (Batched per Month to avoid timeouts)
for m in range(1, 13):
    # Filter days of the month
    col_m = daily_eta_full.filter(ee.Filter.eq('month', m))

    # Convert to Bands (Names: "20250101_ETa_mm", etc.)
    stack_m = col_m.toBands()

    fname = f'SAFER_Daily_Stack_{YEAR}_{m:02d}'

    task = ee.batch.Export.image.toDrive(
        image=stack_m,
        description=fname,
        folder=EXPORT_FOLDER_DAILY,
        fileNamePrefix=fname,
        scale=SCALE,
        region=geometry,
        maxPixels=1e13
        # tileScale removed here
    )
    # Uncomment to run:
    # task.start()

# 2. Monthly Aggregate Stack
monthly_stack = monthly_eta_col.toBands()
task_mon = ee.batch.Export.image.toDrive(
    image=monthly_stack,
    description=f'SAFER_Monthly_Aggregated_{YEAR}',
    folder=EXPORT_FOLDER_MAIN,
    scale=SCALE,
    region=geometry,
    maxPixels=1e13
)
# task_mon.start()

# 3. Annual Map
task_ann = ee.batch.Export.image.toDrive(
    image=annual_eta,
    description=f'SAFER_Annual_Sum_{YEAR}',
    folder=EXPORT_FOLDER_MAIN,
    scale=SCALE,
    region=geometry,
    maxPixels=1e13
)
# task_ann.start()

print("\nTasks defined (commented out). Uncomment 'task.start()' to run.")

In [ ]:
# ==============================================================================
# 7. VISUALIZATION
# ==============================================================================
Map = geemap.Map()
Map.centerObject(geometry, 9)

vis_annual = {'min': 100, 'max': 1200, 'palette': ['#d7191c', '#fdae61', '#ffffbf', '#abdda4', '#2b83ba']}
vis_daily  = {'min': 0, 'max': 8, 'palette': ['#d7191c', '#fdae61', '#ffffbf', '#abdda4', '#2b83ba']}

# 1. Daily Sample (e.g. Jan 15th)
day_sample = daily_eta_full.filterDate(f'{YEAR}-01-15', f'{YEAR}-01-16').first()
if day_sample:
    Map.addLayer(day_sample, vis_daily, f'Daily ETa ({YEAR}-01-15)')

# 2. Annual Sum
Map.addLayer(annual_eta, vis_annual, f'Annual ETa {YEAR}')

Map